## Whole-slide image inference example

This notebook illustrates how to encode tiles from a whole-slide image using a Triton inference server.

If running this notebook on the same machine as the Triton server, prevent TensorFlow from allocating GPU resources.

In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
import tensorflow as tf

assert len(tf.config.list_physical_devices("GPU")) == 0

## Download example data

Download an example whole-slide image and mask pair.

In [ ]:
import pooch

# download whole slide image and corresponding mask
wsi_path = pooch.retrieve(
    fname="TCGA-AN-A0G0-01Z-00-DX1.svs",
    url="https://drive.usercontent.google.com/download?id=19agE_0cWY582szhOVxp9h3kozRfB4CvV&export=download&confirm=t",
    known_hash="d046f952759ff6987374786768fc588740eef1e54e4e295a684f3bd356c8528f",
    path=str(pooch.os_cache("pooch")) + os.sep + "wsi",
)
mask_path = pooch.retrieve(
    fname="TCGA-AN-A0G0-01Z-00-DX1.mask.png",
    url="https://drive.usercontent.google.com/download?id=17GOOHbL8Bo3933rdIui82akr7stbRfta&export=download&confirm=t",
    known_hash="bb657ead9fd3b8284db6ecc1ca8a1efa57a0e9fd73d2ea63ce6053fbd3d65171",
    path=str(pooch.os_cache("pooch")) + os.sep + "wsi",
)

## Create an encoder model

`tf_encoder` creates encoder models using `tensorflow.keras.applications` with configurable input shapes ant types. Selecting a `uint8` dtype allows the encoder model to receive 8-bit inputs instead of float inputs to minimize host-device data transfer. The encoder model is saved into the designated triton server model respository `~/models/` that should be mounted by the triton server container.

In [ ]:
from simple_triton.encoders import tf_encoder

# model parameters
keras_name = "EfficientNetV2S"
model_name = f"{keras_name}.tensorflow"
tile = 224
repository = os.path.join(os.environ["HOME"], "models")
if not os.path.isdir(repository):
    os.mkdir(repository)

# create the model and capture output dimensionality
if not os.path.exists(os.path.join(repository, model_name)):
    dimension_output = tf_encoder(
        repository,
        keras_name,
        model_name,
        input_shape=(tile, tile, 3),
        dtype=tf.uint8,
        pooling="avg",
    )

## Load a model with a minimal configuration

Models served on triton can be configured to control batching, computer resources, and inference optimizations. Models can be configured by providing a configuration during loading with the `TritonModel` class, or by placing a configuration file in the root model directory.

Each triton backend (TensorFlow, ONNX, and python) has unique configuration options and so a configuration class is provided for each. Here we use the `TensorflowConfiguration` class to create a basic configuration. Launching triton server with the `--strict-model-config=false` option enables loading models with a minimal configuration.

In [ ]:
from pprint import pprint
from simple_triton.config import TensorflowConfig

# build a basic configuration specifying only maximum batch size and model name
max_batch_size = 64
name = "EfficientNetV2S.tensorflow"
basic = TensorflowConfig(name, max_batch_size)
pprint(basic.json())

Triton server populates additional fields including information about the input and output dimensions and types and basic optimization for page locking memory used in data transfer.

In [ ]:
from simple_triton.model import TritonModel

# load tensorflow model - set maximum batch size
model = TritonModel(model_name, "localhost:8001")
model.load(config=basic.json())
assert model.is_loaded()
pprint(model.get_config())

## Advanced configuration

Triton can host multiple copies of a model on each GPU using CUDA streams with the `count` option for `InstanceGroup`. This instance group also configures the number of allocated GPUs as well as CPU options. 

Models served with the Tensorflow backend can also benefit from optimizations including mixed precision, XLA compilation, and TensorRT. Mixed precision and TensorRT cannot be used concurrently.

In [ ]:
from simple_triton.config import ConfigBuilder
from simple_triton.config import (
    InstanceGroup,
    TensorflowMixedPrecision,
    TensorflowXla,
    TensorRt,
    TensorflowOptimization,
)

# create a configuration with two model instances per GPU and XLA enabled
instances = InstanceGroup(count=2)
optimization = TensorflowOptimization(
    xla=TensorflowXla(level=2),
    amp=TensorflowMixedPrecision(),
)
xla_amp_config = TensorflowConfig(
    name, max_batch_size, instance_group=instances, optimization=optimization
)
model.load(config=xla_amp_config.json())
assert model.is_loaded()
pprint(model.get_config())

## Run the inference

First, a histomics stream study is created defining the tiles that need to be read based on the whole-slide image, tissue mask, and desired magnification, tile size, and tile overlap. The chunk parameter is used to group tiles during disk reads to maximize throughput. This study initializes a `LargeimagePrefetch` iterator that generates batches of tiles and tile metadata using prefetching.

This iterator is passed to the inference function that is parameterized by the number of tiles per batch, the number of workers, and the maximum number of pending inferences per worker.

In [ ]:
from simple_triton.feature_extraction import inference, study
from simple_triton.tile_iterators import TiffPrefetch
from simple_triton.utils import analyze
from time import time

# slide parameters
batch = 64
magnification = 20.0
chunk = 896
mask_threshold = 0.5

# tile iterator parameters
prefetch = 4
workers = 16  # total number of tile
icc = True  # apply ICC color correction

# create a histomics-stream study from a wsi/mask pair
hs_study = study(
    (wsi_path, mask_path),
    t=(tile, tile),
    chunk=(chunk, chunk),
    objective=magnification,
    mask_threshold=mask_threshold,
)

# inference parameters
limit = 1  # limit on number of pending requests per worker
verbose = True  # display inference statistics and debugging information

# start timer
start = time()

# create tile iterator
iterator = TiffPrefetch(hs_study, np.uint8, icc, batch, prefetch, workers)

# inference
features, metadata, times, failures = inference(
    iterator, model_name, url="localhost:8001", limit=limit, rest=0.0
)

# display elapsed time
print(f"Total elapsed time: {time()-start}")

# analyze performance
analyze(times)

## Write features to .tfr

In [ ]:
from simple_triton.io.tfr_reader import read_record, peek
from simple_triton.io.tfr_writer import write_record

# concatenate features
features = np.concatenate(features[0], axis=0)

# create dummy labels
labels = {"labels": np.random.uniform(size=(10))}

# write to tfrecord
write_record(
    "./triton.tfr", features, metadata, labels, structured=False, precision=tf.float16
)

# get list of .tfr variables for de-serialization
serialized = list(tf.data.TFRecordDataset(["./triton.tfr"]))[0]
variables = peek(serialized)

# verify reading
read_record(serialized, variables, structured=False, precision=tf.float16)

In [ ]:
from google.protobuf import json_format, text_format
import numpy as np
import os
from tritonclient.utils import np_to_triton_dtype
from tritonclient.grpc import model_config_pb2


class DynamicBatching(object):
    """Dynamic batching configuration.

    Dynamic batching allows the aggregation of multiple requests into a single
    inference for optimization.

    Parameters
    ----------
    preferred_batch_size : list
        A list of one or more preferred batch sizes. Default value is [64].
    max_queue_delay_microseconds : int
        The maximimum wait time for dynamic batching. After expiration a request will proceed even
        if the aggregated requests do not meet the preferred batch size. Default value is 0.
    preserve_ordering : bool
        Preserve the order of batches as they are received. Default value is True.

    References
    ----------
    https://github.com/triton-inference-server/server/blob/main/docs/user_guide/model_configuration.md#dynamic-batcher
    """

    def __init__(
        self,
        preferred_batch_size=[64],
        max_queue_delay_microseconds=0,
        preserve_ordering=True,
    ):
        self.config = {
            "PreferredBatchSize": preferred_batch_size,
            "max_queue_delay_microseconds": max_queue_delay_microseconds,
            "preserve_ordering": preserve_ordering,
        }


class ModelInput(object):
    """Model input configuration.

    Configuration of one model input. Pass a list of these inputs for a multi-input model.

    Parameters
    ----------
    name : str
        Model input name.
    shape : list or tuple of int
        The shape of the input not including batch dimension. Variable dimensions are set to -1.
    dtype : numpy.dtype
        The numpy dtype of the model input.
    optional : bool
        Whether this input is optional. Default value is False.
    """

    def __init__(self, name, shape, dtype, optional=False):
        if not isinstance(name, str):
            raise ValueError("name must be str")
        if not isinstance(shape, (list, tuple, np.ndarray)):
            raise ValueError("shape must be type list or type np.ndarray of type int")
        if not all([isinstance(i, (int, np.integer)) for i in shape]):
            raise ValueError("elements of shape must be type int")
        dtype = f"TYPE_{np_to_triton_dtype(dtype().dtype)}"
        input = {
            "name": name,
            "dataType": dtype,
            "dims": shape,
        }
        if optional:
            input["optional":True]
        self.config = input


class ModelOutput(ModelInput):
    """Model output configuration.

    Configuration of one model output. Pass a list of these inputs for a multi-output model.

    Parameters
    ----------
    name : str
        Model output name.
    shape : list or tuple of int
        The shape of the input not including batch dimension. Variable dimensions are set to -1.
    dtype : numpy.dtype
        The numpy dtype of the model input.
    """

    def __init__(self, name, shape, dtype):
        super().__init__(name, shape, dtype, False)


class InstanceGroup(object):
    """Instance group configuration.

    Configures the number and type (CPU/GPU) of processors and the number of model instances
    hosted on each.

    Parameters
    ----------
    count : int
        The number of model instances to run concurrently. Default value is 1.
    kind : str {"cpu", "gpu"}
        Default value of "gpu" specifies that `count` models be hosted on each
        available gpu. Default value is `gpu`.
    gpus : list of int
        If specified, `count` instances will be hosted on each of the listed
        gpus. For example, [0, 1] would specify serving on gpus zero and one.
        Default value of `None` means that `count` instances will be served on
        each available gpu.

    References
    ----------
    https://github.com/triton-inference-server/server/blob/main/docs/user_guide/model_configuration.md#instance-groups
    """

    def __init__(self, count=1, kind="gpu", gpus=None):
        if not isinstance(count, int):
            raise ValueError("`count` must be int.")
        if kind.lower() not in {"cpu", "kind_cpu", "gpu", "kind_gpu"}:
            raise ValueError("`kind` must be one of 'cpu', 'gpu'.")
        elif kind in {"cpu", "kind_cpu"}:
            kind = "KIND_CPU"
        else:
            kind = "KIND_GPU"
        if gpus is not None:
            if not isinstance(gpus, list):
                raise ValueError("argument 'gpus' must be list of type int.")
            if not all([isinstance(inst, int) for inst in gpus]):
                raise ValueError("elements of 'gpus' must be type int.")

        # set count, kind, and optionally GPUs
        instance = {"count": count, "kind": kind}
        if gpus is not None:
            instance["gpus"] = gpus
        self.config = instance


class PythonOptimization(object):
    """Python backend optimization configuration.

    For the python backend page locking of memory used in host-device transfer is the only
    optimization avaialble.

    Parameters
    ----------
    input_pinned : bool
        Page lock memory used to send model inputs. Default value is True.
    output_pinned : bool
        Page lock memory used to recieve model outputs. Default value is True.

    References
    ----------
    https://developer.nvidia.com/blog/how-optimize-data-transfers-cuda-cc/#pinned_host_memory
    """

    def __init__(self, input_pinned=True, output_pinned=True):
        self.config = {
            "inputPinnedMemory": {"enable": input_pinned},
            "outputPinnedMemory": {"enable": output_pinned},
        }


class PythonConfig(object):
    """A model configuration for the python backend.

    This class can generate JSON format dictionaries for use with model loading
    functions, and can save and load configurations in protocol buffer format
    for file-based configuration.

    Parameters
    ----------
    name : str
        Model name as stored in the model repository.
    max_batch_size : int
        The maximum number of samples in a request. Use 0 for a non-batching model.
    input : ModelInput or list
        Model inputs.
    output : ModelOutput or list
        Model outputs.
    instance_group : InstanceGroup
        An instance group configuration defining model resources.
    optimization : PythonOptimization
        Python backend optimization configuration. Default value None enables
        pinned memory by default.
    response_cache : bool
        Whether to cache model input-output pairs. See reference below. Default value
        is False for no caching.

    References
    ----------
    https://github.com/triton-inference-server/server/blob/main/docs/user_guide/response_cache.md
    """

    def __init__(
        self,
        name,
        max_batch_size,
        input=None,
        output=None,
        instance_group=None,
        dynamic_batching=None,
        optimization=None,
        response_cache=False,
    ):
        if not isinstance(name, str):
            raise ValueError("`name` must be type str.")
        if not isinstance(max_batch_size, int):
            raise ValueError("`max_batch_size` must be type int.")
        if input is not None:
            if not isinstance(input, (ModelInput, list)):
                raise ValueError(
                    "`input` must be a ModelInput object or a list of ModelInput objects."
                )
            if isinstance(input, list):
                if not all([isinstance(i, (ModelInput)) for i in input]):
                    raise ValueError("elements of `input` must be a ModelInput object.")
        if output is not None:
            if not isinstance(output, (ModelOutput, list)):
                raise ValueError(
                    "`output` must be a ModelOutput object or a list of ModelOutput objects."
                )
            if isinstance(output, list):
                if not all([isinstance(i, (ModelOutput)) for i in output]):
                    raise ValueError(
                        "elements of `output` must be a ModelOutput object."
                    )
        if instance_group is not None:
            if not isinstance(instance_group, InstanceGroup):
                raise ValueError("`instance_group` must be an InstanceGroup object.")
        if not isinstance(response_cache, bool):
            raise ValueError("`response_cache` must be type bool.")
        self.config = {
            "name": name,
            "versionPolicy": {"latest": {"numVersions": 1}},
            "maxBatchSize": max_batch_size,
            "responseCache": {"enable": response_cache},
            "backend": "python",
        }
        if input is not None:
            self.config["input"] = (
                [i.config for i in input] if input is list else [input.config]
            )
        if output is not None:
            self.config["output"] = (
                [o.config for o in output] if output is list else [output.config]
            )
        if instance_group is not None:
            self.config["instanceGroup"] = [instance_group.config]
        if optimization is not None:
            print(optimization)
            self.config["optimization"] = optimization.config

    def json(self):
        """Return the python model configuration as a JSON dictionary.

        The JSON dictionary can be used with model loading functions.
        """

        return self.config

    def protobuffer(self):
        """Return the python model configuratoin as a protocol buffer."""

        return json_format.ParseDict(self.json(), model_config_pb2.ModelConfig())

    def save(self, path):
        """Save the configuration in protobuffer text (config.pbtxt) format.

        Parameters
        ----------
        path : str
            Path for the output file. File naming is automatic.
        """

        message = json_format.ParseDict(self.json(), model_config_pb2.ModelConfig())
        if not os.path.isdir(path):
            raise ValueError(f"`path` {path} does not exist.")
        with open(os.path.join(path, "config.pbtxt"), "w") as output:
            text_format.PrintMessage(message, output)


class TensorRt(object):
    """TensorRT configuration.

    For use with the TensorFlow and ONNX backends.

    Parameters
    ----------
    precision_mode : str {FP16, FP32}
        Model precision either half-float (FP16) or float (FP32). Default value is "FP16".
    max_cached_engines : int
        The maximum cached TensorRT engines in TensorRT operations. Default value is 100.
    minimum_segment_size : int
        The smallest subgraph size considered for TensorRT optimization. Default value is 3.
    max_workspace_size : int
        The maximum GPU memory available during model execution. Default value is 4 GB.

    References
    ----------
    https://docs.nvidia.com/deeplearning/triton-inference-server/user-guide/docs/user_guide/optimization.html#onnx-with-tensorrt-optimization-ort-trt
    https://docs.nvidia.com/deeplearning/triton-inference-server/user-guide/docs/user_guide/optimization.html#tensorflow-with-tensorrt-optimization-tf-trt
    https://github.com/triton-inference-server/common/blob/main/protobuf/model_config.proto
    """

    def __init__(
        self,
        precision_mode="FP16",
        max_cached_engines=100,
        minimum_segment_size=3,
        max_workspace_size_bytes=4294967296,
    ):
        if precision_mode.upper() not in {"FP16", "FP32"}:
            raise ValueError("`precision_mode` must be one of 'FP16' or 'FP32'.")
        if not isinstance(max_cached_engines, int):
            raise ValueError("`max_cached_engines` must be type int.")
        if not isinstance(minimum_segment_size, int):
            raise ValueError("`minimum_segment_size` must be type int.")
        if not isinstance(max_workspace_size_bytes, int):
            raise ValueError("`max_workspace_size_bytes` must be type int.")
        self.config = {
            "executionAccelerators": {
                "gpuExecutionAccelerator": [
                    {
                        "name": "tensorrt",
                        "parameters": {
                            "precision_mode": f"{precision_mode.upper()}",
                            "max_cached_engines": str(max_cached_engines),
                            "minimum_segment_size": str(minimum_segment_size),
                            "max_workspace_size_bytes": str(max_workspace_size_bytes),
                        },
                    }
                ],
            },
        }


class TensorflowXla(object):
    """Tensorflow XLA graph optimization.

    Sets the level of XLA just in time compilation of tensorflow models.

    Parameters
    ----------
    level : int {-1, 0, 1, 2}
        The optimization level can be off (-1), off but delayed (0), moderate
        optimization(1), or higher optimization (2).
    """

    def __init__(self, level=0):
        if not isinstance(level, int) or not level in {-1, 0, 1, 2}:
            raise ValueError("`level` must be type int with of -1, 0, 1, or 2.")
        self.config = {"graph": {"level": level}}


class TensorflowMixedPrecision(object):
    """Tensorflow automatic mixed precision configuration.

    A configuration that activates automatic mixed precision for half-float inference.
    """

    def __init__(self):
        self.config = {
            "executionAccelerators": {
                "gpuExecutionAccelerator": [{"name": "auto_mixed_precision"}]
            }
        }


class TensorflowOptimization(PythonOptimization):
    """Tensorflow backend optimization configuration.

    The Tensorflow backend supports the memory page locking as well as mixed precision,
    tensorrt, and xla compilation. The default optimization enables page locking and
    mixed precision.

    Parameters
    ----------
    input_pinned : bool
        Page lock memory used to send model inputs. Default value is True.
    output_pinned : bool
        Page lock memory used to recieve model outputs. Default value is True.
    amp : TensorflowMixedPrecision
        An TensorflowMixedPrecision configuration to enable half-float inference.
        Default value is None.
    trt : TensorRt
        A TensorRt configuration to enable reduced precision and operation fusion.
        Default value is None.
    xla : TensorflowXLA
        A TensorflowXla configuration for XLA jit compilation.
        Default value is None.

    References
    ----------
    https://docs.nvidia.com/deeplearning/triton-inference-server/user-guide/docs/user_guide/optimization.html#tensorflow-with-tensorrt-optimization-tf-trt
    https://docs.nvidia.com/deeplearning/triton-inference-server/user-guide/docs/user_guide/optimization.html#tensorflow-automatic-fp16-optimization
    https://docs.nvidia.com/deeplearning/triton-inference-server/user-guide/docs/user_guide/optimization.html#tensorflow-jit-graph-optimizations
    """

    def __init__(
        self,
        input_pinned=True,
        output_pinned=True,
        amp=None,
        trt=None,
        xla=None,
    ):
        super(TensorflowOptimization, self).__init__(input_pinned, output_pinned)
        if amp is not None and trt is not None:
            raise ValueError("`trt` cannot be enabled concurrently with `amp`.")
        if amp is not None:
            if not isinstance(amp, TensorflowMixedPrecision):
                raise ValueError("`amp` must be a TensorflowMixedPrecision object.")
            self.config.update(amp.config)
        if trt is not None:
            if not isinstance(trt, TensorRt):
                raise ValueError("`trt` must be a TensorRt object.")
            self.config.update(trt.config)
        if xla is not None:
            if not isinstance(xla, TensorflowXla):
                raise ValueError("`xla` must be a TensorflowXla object.")
            self.config.update(xla.config)


class TensorflowConfig(PythonConfig):
    """A model configuration for the tensorflow backend.

    This class can generate JSON format dictionaries for use with model loading
    functions, and can save and load configurations in protocol buffer format
    for file-based configuration.

    Parameters
    ----------
    name : str
        Model name as stored in the model repository.
    input : ModelInput or list
        Model inputs.
    output : ModelOutput or list
        Model outputs.
    instance_group : InstanceGroup
        An instance group configuration defining model resources.
    max_batch_size : int
        The maximum number of samples in a request. Use 0 for a non-batching model.
    optimization : TensorflowOptimization
        Python backend optimization configuration. Default value None enables
        pinned memory by default.
    response_cache : bool
        Whether to cache model input-output pairs. See reference below. Default value
        is False for no caching.

    References
    ----------
    https://github.com/triton-inference-server/server/blob/main/docs/user_guide/response_cache.md
    """

    def __init__(
        self,
        name,
        max_batch_size,
        input=None,
        output=None,
        instance_group=None,
        dynamic_batching=None,
        optimization=None,
        response_cache=False,
    ):
        super(TensorflowConfig, self).__init__(
            name=name,
            input=input,
            output=output,
            instance_group=instance_group,
            max_batch_size=max_batch_size,
            dynamic_batching=dynamic_batching,
            response_cache=response_cache,
        )
        if optimization is not None:
            if not isinstance(optimization, TensorflowOptimization):
                raise ValueError(
                    "`optimization` must be a TensorflowOptimization object."
                )
            self.config["optimization"] = optimization.config
        self.config["backend"] = "tensorflow"
        self.config["platform"] = "tensorflow_savedmodel"


def load(path):
    """Load a JSON dictionary configuration from a protobuffer text file.

    Parameters
    ----------
    path : str
        Path for the input file.
    """

    with open(path, "rb") as f:
        protobuf = text_format.Parse(f.read(), model_config_pb2.ModelConfig())
    return json_format.MessageToDict(protobuf)